<a href="https://colab.research.google.com/github/victorcetre/03MIAR-Algoritmos-de-Optimizacion/blob/main/Algoritmos_Victor_Andres_Cetre_Rodriguez_AG3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos - Victor Andres Cetre Rodriguez - AG3

Estudiante: Victor Andres Cetre Rodriguez

URL: https://colab.research.google.com/drive/1p6tCcRO8BdWjDuMlLXw2IYLpc-bLxBZ4?usp=sharing

Github: https://github.com/victorcetre/03MIAR-Algoritmos-de-Optimizacion


##Carga de librerias y datos (TSPLIB - swiss42)

In [ ]:
!pip install requests -q
!pip install tsplib95 -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.1/88.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires tabulate>=0.9, but you have tabulate 0.8.10 which is incompatible.
mapclassify 2.10.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
scikit-image 0.25.2 requires networkx>=3.0, but you have networkx 2.8.8 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.


In [ ]:
import urllib.request
import gzip
import shutil
import os
import tsplib95
import math
import random
import time

#Descargamos el fichero de datos (Matriz de distancias)
file = "swiss42.tsp"

if not os.path.exists(file):
  try:
    urllib.request.urlretrieve("http://comopt.ifi.uni-heidelberg.de/software/TSPLIB95/tsp/swiss42.tsp.gz", file + ".gz")
    with gzip.open(file + ".gz", "rb") as f_in, open(file, "wb") as f_out:
      shutil.copyfileobj(f_in, f_out)
  except Exception as e:
    print("No se pudo descargar el fichero (", e, ").")
    print("Sube manualmente el fichero swiss42.tsp adjunto en la actividad a esta sesion de Colab y vuelve a ejecutar esta celda.")

problem = tsplib95.load(file)
Nodos = list(problem.get_nodes())
print("Nodos:", len(Nodos))
print("Distancia 0->1:", problem.get_weight(0, 1))


No se pudo descargar el fichero ( <urlopen error [Errno 110] Connection timed out> ).
Sube manualmente el fichero swiss42.tsp adjunto en la actividad a esta sesion de Colab y vuelve a ejecutar esta celda.


FileNotFoundError: [Errno 2] No such file or directory: 'swiss42.tsp'

##Funciones básicas

In [ ]:
def longitud_ruta(problem, ruta):
  total = 0
  for i in range(len(ruta)):
    a = ruta[i]
    b = ruta[(i+1) % len(ruta)]
    total += problem.get_weight(a, b)
  return total


def crear_solucion_aleatoria(Nodos):
  resto = Nodos[1:]
  random.shuffle(resto)
  return [Nodos[0]] + resto  #Empezamos y terminamos siempre en el nodo 0


##Búsqueda aleatoria

In [ ]:
def busqueda_aleatoria(problem, Nodos, iteraciones=3000):
  mejor = None
  mejor_long = float("inf")
  for _ in range(iteraciones):
    candidato = crear_solucion_aleatoria(Nodos)
    long_candidato = longitud_ruta(problem, candidato)
    if long_candidato < mejor_long:
      mejor_long = long_candidato
      mejor = candidato
  return mejor, mejor_long


random.seed(1)
sol_aleatoria, long_aleatoria = busqueda_aleatoria(problem, Nodos, 3000)
print("Busqueda aleatoria:", long_aleatoria)


##Búsqueda local (vecindad 2-opt)

In [ ]:
#2-opt: se invierte una sub-lista de la ruta. Se evaluan TODOS los vecinos posibles
#y nos quedamos con el mejor (descenso estricto), hasta que ya no se puede mejorar.
def mejor_vecina_2opt(problem, ruta):
  n = len(ruta)
  mejor = ruta
  mejor_long = longitud_ruta(problem, ruta)
  for i in range(1, n-1):
    for j in range(i+1, n):
      vecina = ruta[:i] + ruta[i:j+1][::-1] + ruta[j+1:]
      long_vecina = longitud_ruta(problem, vecina)
      if long_vecina < mejor_long:
        mejor_long = long_vecina
        mejor = vecina
  return mejor, mejor_long


def busqueda_local(problem, ruta_inicial):
  actual = ruta_inicial
  mejor_long = longitud_ruta(problem, actual)
  while True:
    vecina, long_vecina = mejor_vecina_2opt(problem, actual)
    if long_vecina < mejor_long:
      actual = vecina
      mejor_long = long_vecina
    else:
      return actual, mejor_long


sol_local, long_local = busqueda_local(problem, sol_aleatoria)
print("Busqueda local:", long_local)


##Recocido simulado (Simulated Annealing)

In [ ]:
#El recocido simulado explora con un vecino aleatorio (no exhaustivo, a diferencia
#de la busqueda local) para poder escapar de minimos locales.
def genera_vecina_aleatoria(ruta):
  n = len(ruta)
  i, j = sorted(random.sample(range(1, n), 2))
  return ruta[:i] + ruta[i:j+1][::-1] + ruta[j+1:]


#La probabilidad de aceptar una solucion peor depende de la temperatura T y de
#cuanto empeora la solucion (delta). A mayor T o menor delta, mas facil aceptarla.
def recocido_simulado(problem, ruta_inicial, temp_inicial=100.0, enfriamiento=0.999, iteraciones=20000):
  actual = ruta_inicial[:]
  long_actual = longitud_ruta(problem, actual)
  mejor = actual[:]
  mejor_long = long_actual
  T = temp_inicial

  for _ in range(iteraciones):
    vecina = genera_vecina_aleatoria(actual)
    long_vecina = longitud_ruta(problem, vecina)
    delta = long_vecina - long_actual

    if delta < 0 or random.random() < math.exp(-delta / T):
      actual = vecina
      long_actual = long_vecina
      if long_actual < mejor_long:
        mejor = actual[:]
        mejor_long = long_actual

    T = max(T * enfriamiento, 1e-3)

  return mejor, mejor_long


sol_sa, long_sa = recocido_simulado(problem, sol_aleatoria, 100.0, 0.999, 20000)
print("Recocido simulado:", long_sa)
print("Optimo conocido swiss42: 1273")


##Resumen

In [ ]:
print("Busqueda aleatoria :", long_aleatoria)
print("Busqueda local      :", long_local)
print("Recocido simulado   :", long_sa)
print("Optimo conocido      : 1273")
#La Colonia de Hormigas (ACO) queda fuera de esta entrega: la guia de la actividad
#la marca explicitamente como "no evaluable".
